In [2]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect('loksabha_2024.db')
cursor = conn.cursor()

cursor.execute('DROP TABLE IF EXISTS election_results')

cursor.execute('''
    CREATE TABLE election_results (
        id INTEGER PRIMARY KEY,
        constituency TEXT,
        candidate_name TEXT,
        party TEXT,
        votes_polled INTEGER,
        total_constituency_votes INTEGER
    )
''')

election_data = [
    (1, 'Varanasi', 'Narendra Modi', 'BJP', 612970, 1135000),
    (2, 'Varanasi', 'Ajay Rai', 'INC', 459990, 1135000),
    (3, 'Varanasi', 'Ather Jamal Lari', 'BSP', 33766, 1135000),
    (4, 'Rae Bareli', 'Rahul Gandhi', 'INC', 687507, 1092000),
    (5, 'Rae Bareli', 'Dinesh Pratap Singh', 'BJP', 297477, 1092000),
    (6, 'Rae Bareli', 'Thakur Prasad Maury', 'BSP', 21620, 1092000),
    (7, 'Wayanad', 'Rahul Gandhi', 'INC', 647445, 1085000),
    (8, 'Wayanad', 'Annie Raja', 'CPI', 283476, 1085000),
    (9, 'Wayanad', 'K. Surendran', 'BJP', 141045, 1085000)
]

cursor.executemany('INSERT INTO election_results VALUES (?, ?, ?, ?, ?, ?)', election_data)
conn.commit()
conn.close()
print("✅ Lok Sabha Election Database populated successfully!")

✅ Lok Sabha Election Database populated successfully!


In [3]:
conn = sqlite3.connect('loksabha_2024.db')

query = '''
    SELECT constituency, candidate_name, party, votes_polled, total_constituency_votes
    FROM election_results
    ORDER BY constituency, votes_polled DESC
'''
df = pd.read_sql_query(query, conn)

votes_polled = df['votes_polled'].to_numpy()
total_votes = df['total_constituency_votes'].to_numpy()
df['vote_share_pct'] = np.round((votes_polled / total_votes) * 100, 2)

df['rank'] = df.groupby('constituency')['votes_polled'].rank(ascending=False, method='min').astype(int)

winners = df[df['rank'] == 1].copy()
runners_up = df[df['rank'] == 2].copy()

analysis_df = pd.merge(
    winners,
    runners_up[['constituency', 'candidate_name', 'votes_polled']],
    on='constituency',
    suffixes=('_winner', '_runner_up')
)

analysis_df['victory_margin'] = analysis_df['votes_polled_winner'] - analysis_df['votes_polled_runner_up']

print("--- CONSTITUENCY WINNERS & MARGINS ---")
print(analysis_df[['constituency', 'candidate_name_winner', 'party', 'victory_margin', 'vote_share_pct']])

seat_tally = winners['party'].value_counts().reset_index()
seat_tally.columns = ['Party', 'Seats Won']

print("\n--- FINAL PARTY SEAT TALLY ---")
print(seat_tally)

conn.close()

--- CONSTITUENCY WINNERS & MARGINS ---
  constituency candidate_name_winner party  victory_margin  vote_share_pct
0   Rae Bareli          Rahul Gandhi   INC          390030           62.96
1     Varanasi         Narendra Modi   BJP          152980           54.01
2      Wayanad          Rahul Gandhi   INC          363969           59.67

--- FINAL PARTY SEAT TALLY ---
  Party  Seats Won
0   INC          2
1   BJP          1
